# SAGE — YOLOv8n Custom Accident Detection Training

This notebook trains a YOLOv8n model for Indian road accident detection.
Class mapping (7 classes):
- 0: person
- 1: car
- 2: motorcycle
- 3: bus
- 4: truck
- 5: bicycle
- 6: fallen_person

**Output:** `best.onnx` — place at `public/models/best.onnx` in the Next.js project.

In [ ]:
# 1. Install dependencies
!pip install -q ultralytics roboflow onnx onnxruntime

In [ ]:
# 2. Clone or mount your dataset
# Option A: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Option B: Use Roboflow (uncomment below)
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_KEY")
# project = rf.workspace("your-workspace").project("your-project")
# version = project.version(1)
# dataset = version.download("yolov8")
# DATASET_DIR = dataset.location

In [ ]:
# 3. Configure dataset path
# Point this to your merged dataset directory (see merge_datasets.py)
# The directory must contain: train/images, train/labels, val/images, val/labels
DATASET_DIR = "/content/drive/MyDrive/sage-dataset/merged"  # <-- UPDATE THIS

import os
assert os.path.exists(DATASET_DIR), f"Dataset not found at {DATASET_DIR}"
print(f"Dataset: {DATASET_DIR}")
print(f"Train images: {len(os.listdir(os.path.join(DATASET_DIR, 'train', 'images')))}")
print(f"Val images: {len(os.listdir(os.path.join(DATASET_DIR, 'val', 'images')))}")

In [ ]:
# 4. Write datasets.yaml
yaml_content = f"""
train: {DATASET_DIR}/train/images
val: {DATASET_DIR}/val/images
test: {DATASET_DIR}/test/images

nc: 7
names:
  0: person
  1: car
  2: motorcycle
  3: bus
  4: truck
  5: bicycle
  6: fallen_person
"""

with open("datasets.yaml", "w") as f:
    f.write(yaml_content)
print("Written datasets.yaml")

In [ ]:
# 5. Train YOLOv8n
from ultralytics import YOLO

# Start from pretrained COCO weights — our classes overlap significantly
model = YOLO("yolov8n.pt")

results = model.train(
    data="datasets.yaml",
    epochs=150,
    imgsz=640,
    batch=16,
    lr0=0.01,
    patience=30,
    device=0,
    project="runs/train",
    name="sage_accident",
    # Augmentations tuned for Indian road conditions
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    optimizer="auto",
    cos_lr=True,
    warmup_epochs=3,
    save_period=10,
    workers=2,
    verbose=True,
)
print(f"Training complete! Best weights at: {results.save_dir}/weights/best.pt")

In [ ]:
# 6. Evaluate on validation set
model = YOLO("runs/train/sage_accident/weights/best.pt")

metrics = model.val(data="datasets.yaml", device=0)

print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

names = metrics.names
for i, (p, r, ap50, ap) in enumerate(
    zip(metrics.box.p, metrics.box.r, metrics.box.ap50, metrics.box.ap)
):
    print(f"  {names[i]:20s}  P={p:.3f}  R={r:.3f}  AP50={ap50:.3f}  AP={ap:.3f}")

In [ ]:
# 7. Export to ONNX
model = YOLO("runs/train/sage_accident/weights/best.pt")

onnx_path = model.export(
    format="onnx",
    imgsz=640,
    simplify=True,
    opset=13,
    dynamic=False,
    half=False,
)

print(f"ONNX exported to: {onnx_path}")

# Copy to a convenient location for download
import shutil
shutil.copy(onnx_path, "/content/best.onnx")
print("Copied to /content/best.onnx — download this file.")
print("Place it at: public/models/best.onnx in your Next.js project.")

In [ ]:
# 8. (Optional) INT8 quantization for smaller model
try:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic(
        model_input="/content/best.onnx",
        model_output="/content/best_int8.onnx",
        weight_type=QuantType.QUInt8,
    )
    print("INT8 quantized model: /content/best_int8.onnx")
except Exception as e:
    print(f"Quantization skipped: {e}")

In [ ]:
# 9. Verify ONNX model loads correctly
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession("/content/best.onnx")
input_info = session.get_inputs()[0]
output_info = session.get_outputs()[0]

print(f"Input:  {input_info.name}, shape={input_info.shape}, type={input_info.type}")
print(f"Output: {output_info.name}, shape={output_info.shape}, type={output_info.type}")

# Quick sanity check with dummy input
dummy = np.random.randn(1, 3, 640, 640).astype(np.float32)
result = session.run(None, {input_info.name: dummy})
print(f"Inference OK — output shape: {result[0].shape}")
print("\nDone! Download /content/best.onnx and place at public/models/best.onnx")